In [3]:
import json
import time

# External Dependencies:
import boto3  # AWS SDK for Python
from botocore.exceptions import ClientError  # for error handling
from IPython.display import Markdown, display  # Notebook display utilities

In [4]:
session = boto3.Session()
region = session.region_name
bedrock = boto3.client(service_name="bedrock-runtime", region_name=region)

In [5]:
prompt_data = """
Command: Write an email from Bob, Customer Service Manager, to the customer "John Doe" 
who provided negative feedback on the service provided by our customer support 
engineer"""

In [14]:
body = json.dumps({
    
        "messages": [{"role": "user", "content": [{"type": "text", "text": prompt_data}]}],
        "maxTokenCount":4096,
        "stopSequences":[],
        "temperature":0,
        "topP":0.9,
        
        
    })

In [15]:

modelId = 'moonshotai.kimi-k2.5'
accept = 'application/json'
contentType = 'application/json'
outputText = "\n"
full_response = ""
try:

    response = bedrock.invoke_model_with_response_stream(body=body, modelId=modelId, accept=accept, contentType=contentType)
    stream = response.get('body')
    i = 1
    if stream:
        for event in stream:
            chunk = event.get('chunk')
            if chunk:
                chunk_data = json.loads(chunk.get('bytes').decode('utf-8'))
                choices = chunk_data.get('choices', [])
                if choices:
                    delta_content = choices[0].get('delta', {}).get('content', '')
                    full_response += delta_content
    print(full_response)
except ClientError as error:
    if error.response["Error"]["Code"] == "AccessDeniedException":
        print(
            f"\x1b[41m{error.response['Error']['Code']}: {error.response['Error']['Message']}\x1b[0m"
        )
        print("Please ensure you have the necessary permissions for Amazon Bedrock.")
    else:
        raise error

 Subject: Re: Your Recent Feedback – We're Here to Make This Right

Dear Mr. Doe,

Thank you for taking the time to share your feedback regarding your recent experience with our customer support engineer. I sincerely apologize that the service you received did not meet the standards you expected and that we strive to deliver.

As the Customer Service Manager, I want you to know that your concerns have been brought to my attention, and I am personally reviewing what occurred during your interaction. We take all feedback seriously, as it helps us identify areas where we can improve.

I would welcome the opportunity to discuss your experience in more detail and work toward a resolution. Please feel free to reply to this email or call me directly at [phone number] at your earliest convenience. I am committed to ensuring we address your concerns and restore your confidence in our support services.

Thank you again for bringing this matter to our attention. We value your business and hope to